# Example 15.1 / 2: What AMPLbot can and cannot do

Companion notebook for **15.1 AMPLbot in ChatGPT** in the AMPL lab Documentation, section "What AMPLbot can and cannot do". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## What AMPLbot can and cannot do

| It can help you | It does not establish |
|---|---|
| Learn AMPL syntax and modeling patterns | That the business requirements are complete |
| Draft LP, MIP, NLP, MINLP, CP, and multi-objective formulations | That a chosen formulation is mathematically equivalent to your intent |
| Build AMPL+Python workflows with `amplpy` | That generated code ran successfully |
| Investigate infeasibility, unboundedness, and solver errors | That a returned solution is acceptable for production |
| Suggest reformulations and solver options | That an option exists or is suitable for your solver version |

AMPLbot does not execute the optimization code it writes. Run the model yourself and apply the checks in "Check a Generated Model by Running It" above.

The right-hand column of the table lists what you must establish yourself. Its last row is
the easiest to test: a suggested option that does not exist is rejected by the solver. Here
`mip_gap` is a plausible but unknown name for HiGHS:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;                  # steel products
param rate {PROD} > 0;     # tons per hour
param profit {PROD};       # dollars per ton
param market {PROD} >= 0;  # maximum sales, tons
param avail >= 0;          # rolling hours available

var Make {p in PROD} integer >= 0, <= market[p];  # integer tons: makes the model a MIP

maximize Total_Profit: sum {p in PROD} profit[p] * Make[p];  # total profit, dollars

# rolling hours used fit within avail
subject to Time: sum {p in PROD} (1 / rate[p]) * Make[p] <= avail;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils"]
ampl.param["rate"] = {"bands": 200, "coils": 140}
ampl.param["profit"] = {"bands": 25, "coils": 30}
ampl.param["market"] = {"bands": 6000, "coils": 4000}
ampl.param["avail"] = 40

In [5]:
# run.py
ampl.option["highs_options"] = "mip_gap=0.01"
ampl.solve(solver="highs")
print(ampl.solve_result)

HiGHS 1.14.0: Unknown option or invalid key "mip_gap"
failure


### Another way in amplpy

The HiGHS option list names the relative MIP gap `mip:gap`, with aliases `mipgap`,
`mip:relgaptol`, `relgaptol` and `mip_rel_gap`. With the documented name the same model solves:

In [6]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [7]:
# data.py
ampl.set["PROD"] = ["bands", "coils"]
ampl.param["rate"] = {"bands": 200, "coils": 140}
ampl.param["profit"] = {"bands": 25, "coils": 30}
ampl.param["market"] = {"bands": 6000, "coils": 4000}
ampl.param["avail"] = 40

In [8]:
# run.py
ampl.option["highs_options"] = "mip:gap=0.01"
ampl.solve(solver="highs")
print(ampl.solve_result)

HiGHS 1.14.0:   mip:gap = 0.01
HiGHS 1.14.0: optimal solution; objective 192000
1 simplex iterations
1 branching nodes
solved
